<a href="https://colab.research.google.com/github/Manny2282/CFB-Client-Data/blob/main/CFB_Client_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import os
from collections import defaultdict

In [ ]:
!pip install supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.6 MB/s eta 0:00:00


In [ ]:
import requests
import os
from collections import defaultdict
from supabase import create_client, Client
from google.colab import userdata

cfb_api = 'xrkNNrzME76a3MO+yJuEKOkPD+2+LAwdKKQFG151XY3W/XCz85IXoky8DGO9zR+y'

# Supabase URL and Key from Colab secrets
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

# Supabase client
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# URL for the College Football Data API
CFB_BASE_URL = "https://api.collegefootballdata.com"

# HEADERS for API requests
HEADERS = {"Authorization": f"Bearer {cfb_api}"}

In [ ]:
print("SUPABASE_URL =", SUPABASE_URL)
print("SUPABASE_KEY is None? ", SUPABASE_KEY is None)

SUPABASE_URL = https://yugzhcfqzisdirdtxcyk.supabase.co
SUPABASE_KEY is None?  False


In [ ]:
import socket
host = "yugzhcfqzisdirdtxcyk.supabase.co"
print("Resolving host:", host)
print("IP:", socket.gethostbyname(host))

Resolving host: yugzhcfqzisdirdtxcyk.supabase.co
IP: 104.18.38.10


In [ ]:
# Postion Groups and Weight for PPA
postion_groups = {
    "QB":["QB"],
    "RB":["RB","FB"],
    "WR":["WR"],
    "TE":["TE"],
    "OL":["C","RT","LT","RG","LG"],
    "DL":["NT","DE","DT"],
    "LB":["OLB","ILB","MLB"],

    "DB":["CB","SS","FS"],
}
weights = {
    "QB":(0.75,0.25),
    "RB":(0.65,0.35),
    "WR":(0.65,0.35),
    "TE":(0.65,0.35),
    "OL":(0.5,0.5),
    "DL":(0.7,0.3),
    "LB":(0.7,0.3),
    "DB":(0.7,0.3)
}

In [ ]:
# Getting endpoints
def fetch(endpoint, params):
  r = requests.get(f"{CFB_BASE_URL}{endpoint}", headers = HEADERS ,params = params, timeout = 30 )
  r.raise_for_status()
  return r.json()
# Putting postions in groups
def group_for_postion(pos):
  for group, members in postion_groups.items():
    if pos in members:
      return group
# Get percent rank
def percentile_rank(values, target):
  if not values:
    return 0.0
  bor = sum(
      1 for v in values
      if v <= target)
  return bor / len(values)


In [ ]:
SEASON = 2026 # Define the season for data fetching

In [ ]:
SEASON = 2026 # Define the season for data fetching, using the current year as specified by the user
# PLAYER_ID_TO_TEST = "190111" # Removed playerId for initial fetch to see if it resolves the 400 error
print(f'Fetching ALL sample player season stats entries for SEASON {SEASON} to inspect their structure...')
sample_stats = fetch("/stats/player/season", {"year": SEASON})

if sample_stats:
    print(f"Fetched {len(sample_stats)} player season stats entries. Displaying the first 5 entries:")
    # Display first 5 entries to see the structure of all player stats
    for i, entry in enumerate(sample_stats):
        if i >= 5: break
        display(entry)
else:
    print(f"No sample player season stats found for season {SEASON}")

# NOTE: After inspecting the output, we will verify the main function now correctly extracts passing and rushing yards.

Fetching ALL sample player season stats entries for SEASON 2026 to inspect their structure...
Fetched 80698 player season stats entries. Displaying the first 5 entries:


{'season': 2026,
 'playerId': '245322',
 'player': 'Marcus Patterson',
 'position': 'DL',
 'team': 'Kennesaw State',
 'conference': 'Conference USA',
 'category': 'defensive',
 'statType': 'PD',
 'stat': '0'}

{'season': 2026,
 'playerId': '245322',
 'player': 'Marcus Patterson',
 'position': 'DL',
 'team': 'Kennesaw State',
 'conference': 'Conference USA',
 'category': 'defensive',
 'statType': 'QB HUR',
 'stat': '0'}

{'season': 2026,
 'playerId': '245322',
 'player': 'Marcus Patterson',
 'position': 'DL',
 'team': 'Kennesaw State',
 'conference': 'Conference USA',
 'category': 'defensive',
 'statType': 'SACKS',
 'stat': '0'}

{'season': 2026,
 'playerId': '245322',
 'player': 'Marcus Patterson',
 'position': 'DL',
 'team': 'Kennesaw State',
 'conference': 'Conference USA',
 'category': 'defensive',
 'statType': 'SOLO',
 'stat': '0'}

{'season': 2026,
 'playerId': '245322',
 'player': 'Marcus Patterson',
 'position': 'DL',
 'team': 'Kennesaw State',
 'conference': 'Conference USA',
 'category': 'defensive',
 'statType': 'TD',
 'stat': '0'}

In [ ]:
def main():
  print("Getting roster...")
  roster = fetch("/roster", {"year": SEASON})
  print(f"Fetched {len(roster)} roster entries.")
  if roster:
      print("Sample roster entry:", roster[0])

  print("getting PPA")
  ppa = fetch("/ppa/players/season", {"year": SEASON })
  print(f"Fetched {len(ppa)} PPA entries.")

  print("Getting usage...")
  usage = fetch("/player/usage", {"year": SEASON})
  print(f"Fetched {len(usage)} usage entries.")

  print("Getting player season stats...")
  raw_player_season_stats = fetch("/stats/player/season", {"year": SEASON})
  print(f"Fetched {len(raw_player_season_stats)} player season stats entries.")

  # Reconstruct player_season_stats_by_id to group all stats for each player
  player_season_stats_by_id = defaultdict(list)
  for stat_entry in raw_player_season_stats:
      player_id = stat_entry.get("playerId")
      if player_id:
          player_season_stats_by_id[player_id].append(stat_entry)

  # lists to dictionaries for easy lookup
  ppa_by_id = {p["id"]: p for p in ppa}
  usage_by_id = {u["id"]: u for u in usage}

  # player and stats rows
  players_row = []
  stats_row = []
  injury_row = []
  rating_row = []
  group_ppa_values = defaultdict(list)
  group_usage_values = defaultdict(list)
  merged = []

  for player in roster:
    athlete_id = player["id"]
    position = player.get("position")
    group = group_for_postion(position)
    if not group:
      continue

    ppa_row = ppa_by_id.get(athlete_id, {})
    usage_row = usage_by_id.get(athlete_id, {})
    all_player_stats = player_season_stats_by_id.get(athlete_id, [])

    avg_ppa_overall = (ppa_row.get("averagePPA") or {}).get("all", 0.0)
    usage_pct = (usage_row.get("usage") or {}).get("overall", 0.0)

    # Initialize a dictionary to store all stats for the current player
    player_stats_data = {}

    # Extract specific stats from all_player_stats and store them dynamically
    for stat_item in all_player_stats:
        category = stat_item.get("category")
        stat_type = stat_item.get("statType")
        stat_value_str = stat_item.get("stat")

        if category and stat_type and stat_value_str is not None:
            try:
                # Try converting to int, if fails, try float
                if '.' in stat_value_str: # Heuristic to check if it might be a float
                    stat_value = float(stat_value_str)
                else:
                    stat_value = int(stat_value_str)
            except ValueError:
                stat_value = stat_value_str # Keep as string if conversion fails

            # Create a unique key for the stat
            stat_key = f"{category}_{stat_type}"
            player_stats_data[stat_key] = stat_value

    players_row.append({
        "athlete id": athlete_id,
        "first_name": player.get("firstName"),
        "last_name": player.get("lastName"),
        "team": player.get("team"),
        "position": position,
        "class_year": player.get("year"),
        "height": player.get("height"),
        "weight": player.get("weight")
        })
    stats_row.append({
        "athlete id": athlete_id,
        "season": SEASON,
        "usage_pct": usage_pct,
        "avg_ppa_overall": avg_ppa_overall,
        "plays": usage_row.get("plays"),
        **player_stats_data # Unpack all collected stats here
    })

    injury_row.append({
        "athlete id": athlete_id,
        "current injury report": None,
        "injury history": None,
    })

    group_ppa_values[group].append(avg_ppa_overall)
    group_usage_values[group].append(usage_pct)
    merged.append((athlete_id, group, avg_ppa_overall, usage_pct))

  # rating equation
  for athlete_id, group, ppa_value, usage_val in merged:
    ppa_pct = percentile_rank(group_ppa_values[group], ppa_value)
    usage_pct_rank = percentile_rank(group_usage_values[group], usage_val)
    w_ppa, w_usage = weights[group]
    rating = round(w_ppa * ppa_pct + w_usage * usage_pct_rank, 3)

    rating_row.append({
        "athlete_id": athlete_id,
        "season": SEASON,
        "position_group": group,
        "ppa_percentile": ppa_pct,
        "rating": rating
    })
  return players_row, stats_row, rating_row, injury_row

In [ ]:
passing_stat_types = set()
rushing_stat_types = set()
other_categories = set()
example_passing_stats = []
example_rushing_stats = []

for entry in sample_stats:
    category = entry.get('category')
    stat_type = entry.get('statType')

    if category == 'passing':
        passing_stat_types.add(stat_type)
        if len(example_passing_stats) < 5 and stat_type: # Get up to 5 examples
            example_passing_stats.append(entry)
    elif category == 'rushing':
        rushing_stat_types.add(stat_type)
        if len(example_rushing_stats) < 5 and stat_type: # Get up to 5 examples
            example_rushing_stats.append(entry)
    else:
        other_categories.add(category)

print("Unique statTypes for 'passing' category:", passing_stat_types)
print("Example 'passing' stats:")
for example in example_passing_stats:
    display(example)

print("\nUnique statTypes for 'rushing' category:", rushing_stat_types)
print("Example 'rushing' stats:")
for example in example_rushing_stats:
    display(example)

print("\nOther categories found:", other_categories)

Unique statTypes for 'passing' category: {'PCT', 'ATT', 'YPA', 'YDS', 'COMPLETIONS', 'INT', 'TD'}
Example 'passing' stats:


{'season': 2026,
 'playerId': '4431633',
 'player': 'Will Crowder',
 'position': 'QB',
 'team': 'Troy',
 'conference': 'Sun Belt',
 'category': 'passing',
 'statType': 'ATT',
 'stat': '24'}

{'season': 2026,
 'playerId': '4431633',
 'player': 'Will Crowder',
 'position': 'QB',
 'team': 'Troy',
 'conference': 'Sun Belt',
 'category': 'passing',
 'statType': 'COMPLETIONS',
 'stat': '12'}

{'season': 2026,
 'playerId': '4431633',
 'player': 'Will Crowder',
 'position': 'QB',
 'team': 'Troy',
 'conference': 'Sun Belt',
 'category': 'passing',
 'statType': 'INT',
 'stat': '0'}

{'season': 2026,
 'playerId': '4431633',
 'player': 'Will Crowder',
 'position': 'QB',
 'team': 'Troy',
 'conference': 'Sun Belt',
 'category': 'passing',
 'statType': 'PCT',
 'stat': '0.500'}

...

{'season': 2026,
 'playerId': '518623',
 'player': 'Isaiah Jones',
 'position': 'TE',
 'team': 'Maine',
 'conference': 'Coastal Athletic',
 'category': 'rushing',
 'statType': 'LONG',
 'stat': '0'}

{'season': 2026,
 'playerId': '518623',
 'player': 'Isaiah Jones',
 'position': 'TE',
 'team': 'Maine',
 'conference': 'Coastal Athletic',
 'category': 'rushing',
 'statType': 'TD',
 'stat': '0'}

{'season': 2026,
 'playerId': '518623',
 'player': 'Isaiah Jones',
 'position': 'TE',
 'team': 'Maine',
 'conference': 'Coastal Athletic',
 'category': 'rushing',
 'statType': 'YDS',
 'stat': '0'}

{'season': 2026,
 'playerId': '518623',
 'player': 'Isaiah Jones',
 'position': 'TE',
 'team': 'Maine',
 'conference': 'Coastal Athletic',
 'category': 'rushing',
 'statType': 'YPC',
 'stat': '0.0'}


Other categories found: {'receiving', 'defensive', 'kickReturns', 'punting', 'puntReturns', 'fumbles', 'interceptions', 'kicking'}


In [ ]:
def upserting_batchs(table, rows, batch_size = 500):
  for i in range(0, len(rows), batch_size):
    batch = rows[i:i + batch_size]
    supabase.table(table).upsert(batch).execute()

In [ ]:
if __name__ == "__main__":
  players_row, stats_row, rating_row, injury_row = main() # Call main

  # Function to replace spaces in dictionary keys with underscores for Supabase compatibility
  def fix_keys_for_supabase(data_rows):
      fixed_data = []
      for row in data_rows:
          fixed_row = {k.replace(' ', '_'): v for k, v in row.items()}
          fixed_data.append(fixed_row)
      return fixed_data

  # Apply the key fixing to rows that are likely to have keys with spaces
  players_row = fix_keys_for_supabase(players_row)
  stats_row = fix_keys_for_supabase(stats_row)
  injury_row = fix_keys_for_supabase(injury_row)

  # Deduplicate players_row based on 'athlete_id'
  unique_players = {player['athlete_id']: player for player in players_row}
  players_row = list(unique_players.values())

  # Deduplicate stats_row based on 'athlete_id' and 'season' before upserting
  unique_stats = {(stat['athlete_id'], stat['season']): stat for stat in stats_row}
  stats_row = list(unique_stats.values())

  # Deduplicate rating_row based on 'athlete_id', 'season', and 'position_group'
  unique_ratings = {(r['athlete_id'], r['season'], r['position_group']): r for r in rating_row}
  rating_row = list(unique_ratings.values())

  # upsert players_row with 'on_conflict'
  print(f"Upserting {len(players_row)} players with explicit 'athlete_id' conflict resolution...")
  BATCH_SIZE = 500
  for i in range(0, len(players_row), BATCH_SIZE):
      batch = players_row[i:i + BATCH_SIZE]
      supabase.table("players").upsert(batch, on_conflict='athlete_id').execute()

  print(f"Upserting {len(stats_row)} season stat rows...")
  if stats_row:
      print("Sample stats_row after key fixing:", stats_row[0])
  upserting_batchs("players_season_stats", stats_row)

  print(f"Upserting {len(rating_row)} ratings...")
  upserting_batchs("player_ratings", rating_row)

  # Commenting out injury_row upsert due to APIError: table 'public.player_injuries' not found.
  # print(f"Upserting {len(injury_row)} injury rows...")
  # upserting_batchs("player_injuries", injury_row)

  print("Done with main data processing and upserts.")

  # for row in get_freshman_above(0.80):
  #   p = row["players"]
  #   print(f"{row['rating']} {p['first_name']} {p['last_name']}"
  #         f"({p['postion']}, [p{{'team'}}])")

Getting roster...
Fetched 30639 roster entries.
Sample roster entry: {'id': '158607', 'firstName': 'Maurice', 'lastName': 'Smith', 'team': 'Tennessee Tech', 'weight': 195, 'height': None, 'jersey': None, 'year': 1, 'position': 'TE', 'homeCity': 'Athens', 'homeState': 'TN', 'homeCountry': '', 'homeLatitude': None, 'homeLongitude': None, 'homeCountyFIPS': None, 'recruitIds': []}
getting PPA
Fetched 3581 PPA entries.
Getting usage...
Fetched 3581 usage entries.
Getting player season stats...
Fetched 80698 player season stats entries.
Upserting 11954 players with explicit 'athlete_id' conflict resolution...
Upserting 11954 season stat rows...
Sample stats_row after key fixing: {'athlete_id': '158607', 'season': 2026, 'usage_pct': 0.0, 'avg_ppa_overall': 0.0, 'plays': None}
Upserting 11954 ratings...
Done with main data processing and upserts.


In [ ]:
unique_stat_columns = set()
for stat_entry in sample_stats:
    category = stat_entry.get("category")
    stat_type = stat_entry.get("statType")
    if category and stat_type:
        unique_stat_columns.add(f"{category}_{stat_type}")

# Sort the columns for consistent output
sorted_unique_stat_columns = sorted(list(unique_stat_columns))

print("Columns to consider adding to your players_season_stats table:")
for col in sorted_unique_stat_columns:
    print(col)

Columns to consider adding to your players_season_stats table:
defensive_PD
defensive_QB HUR
defensive_SACKS
defensive_SOLO
defensive_TD
defensive_TFL
defensive_TOT
fumbles_FUM
fumbles_LOST
fumbles_REC
interceptions_AVG
interceptions_INT
interceptions_TD
interceptions_YDS
kickReturns_AVG
kickReturns_LONG
kickReturns_NO
kickReturns_TD
kickReturns_YDS
kicking_FGA
kicking_FGM
kicking_LONG
kicking_PCT
kicking_PTS
kicking_XPA
kicking_XPM
passing_ATT
passing_COMPLETIONS
passing_INT
passing_PCT
passing_TD
passing_YDS
passing_YPA
puntReturns_AVG
puntReturns_LONG
puntReturns_NO
puntReturns_TD
puntReturns_YDS
punting_In 20
punting_LONG
punting_NO
punting_TB
punting_YDS
punting_YPP
receiving_LONG
receiving_REC
receiving_TD
receiving_YDS
receiving_YPR
rushing_CAR
rushing_LONG
rushing_TD
rushing_YDS
rushing_YPC
